# Unidad 3 : Creación de un RAG para que el agente reciba información

In [ ]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-retrievers-bm25 datasets nest_asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(
    model="gemini-3.1-flash-lite",
    api_key=userdata.get("GEMINI_API_KEY"),
    temperature=0.7,
    max_tokens=1024,
)

Para utilizar un RAG primero debemos crear el modelo que va a estar consultando dicho RAG.

In [ ]:
import datasets
from llama_index.core.schema import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        text="\n".join([
            f"Name: {guest_dataset['name'][i]}",
            f"Relation: {guest_dataset['relation'][i]}",
            f"Description: {guest_dataset['description'][i]}",
            f"Email: {guest_dataset['email'][i]}"
        ]),
        metadata={"name": guest_dataset['name'][i]}
    )
    for i in range(len(guest_dataset))
]

print(len(docs), "guests loaded")

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.32kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3 [00:00<?, ? examples/s]

3 guests loaded


Luego cargamos el documento que se va autilizar para consulta en un objeto de tipo Document.

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.retrievers.bm25 import BM25Retriever

bm25_retriever = BM25Retriever.from_defaults(nodes=docs, similarity_top_k=3)

def get_guest_info_retriever(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.retrieve(query)
    if results:
        return "\n\n".join([doc.text for doc in results[:3]])
    else:
        return "No matching guest information found."

# Initialize the tool
guest_info_tool = FunctionTool.from_defaults(get_guest_info_retriever)

DEBUG:bm25s:Building index from IDs objects


Luego definimos la herramienta con la que el agente va a poder consultar el documento que cargamos.

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

# Create Alfred, our gala agent, with the guest info tool
alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool],
    llm=llm,
)

# Example query Alfred might receive during the gala
response = await alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")

print("🎩 Alfred's Response:")
print(response)

response = await alfred.run("What do you know about Dr. Nikola Tesla?")
print(response)

🎩 Alfred's Response:
Lady Ada Lovelace is your best friend and an esteemed mathematician. She is widely celebrated for her pioneering work in computing and is often considered the world's first computer programmer, thanks to her groundbreaking contributions to Charles Babbage's Analytical Engine.
Dr. Nikola Tesla is an old friend of yours from your university days. He is an innovative inventor who has recently patented a new wireless energy transmission system. 

If you're planning to speak with him, he is quite passionate about pigeons, which could be an excellent topic for small talk. You can reach him at nikola.tesla@gmail.com.


Para finalizar se inicializa el agente y se confirma que si utilice el documento para hacer sus consultas correspondientes.

In [ ]:
!pip install -q -U ddgs

from ddgs import DDGS

def web_search(query: str) -> str:
    """Searches the web with DuckDuckGo and returns the top results with titles, links, and snippets."""
    try:
        results = DDGS().text(query, max_results=5)
    except Exception as e:
        return f"Search failed: {e}"
    if not results:
        return "No results found."
    return "\n\n".join(f"{r['title']}\n{r['href']}\n{r['body']}" for r in results)

search_tool = FunctionTool.from_defaults(web_search, name="web_search")

# quick test
print(web_search("Who is the current President of France?")[:500])

Emmanuel Macron - Wikipedia
https://en.wikipedia.org/wiki/Emmanuel_Macron
Emmanuel Jean-Michel Frédéric Macron[b] (born 21 December 1977) is a French politician who has served as President of France since 2017. He has been a member of Renaissance since founding the party in 2016.

President of France - Wikipedia
https://en.wikipedia.org/wiki/President_of_France
The current president is Emmanuel Macron, who succeeded François Hollande on 14 May 2017 following the 2017 presidential election, and w


Ahora se le va a dar acceso al agente a herramientas  adicionales para que pueda complementar la información que encuentra en sus consultas de RAG. Para empezar se le da acceso a una herramienta para tener acceso a internet.

In [ ]:
import random

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20},
    ]
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

weather_info_tool = FunctionTool.from_defaults(get_weather_info)

print(get_weather_info("Paris"))

Weather in Paris: Rainy, 15°C


Luego se le da acceso a una herramienta simple para consultar registros (falsos) del clima de una ubicación en particular.

In [ ]:
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

hub_stats_tool = FunctionTool.from_defaults(get_hub_stats)

print(get_hub_stats("facebook"))

The most downloaded model by facebook is facebook/contriever with 8,302,069 downloads.


Para finalizar, creamos una herramienta para que el agente pueda acceder a los modelos más populares en Hugging Face.

In [ ]:
alfred = AgentWorkflow.from_tools_or_functions(
    [search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
)

response = await alfred.run("What is Facebook and what's their most popular model?")
print("🎩 Alfred's Response:")
print(response)

🎩 Alfred's Response:
### What is Facebook (Meta)?

**Facebook** is a social media platform launched in 2004 by Mark Zuckerberg and his colleagues at Harvard University. It quickly grew to become one of the most widely used websites in the world, allowing users to connect with friends, family, and communities.

In 2021, the parent company of Facebook rebranded itself as **Meta Platforms, Inc.** (commonly known as **Meta**). This change was made to reflect the company's broader ambitions beyond just social media, including its focus on the "metaverse" (virtual and augmented reality) and significant investments in artificial intelligence (AI).

Today, Meta owns several major platforms and services, including:
*   **Facebook**
*   **Instagram**
*   **WhatsApp**
*   **Messenger**
*   **Threads**

Meta is also a major player in the field of artificial intelligence research and development, contributing heavily to the open-source community by releasing various AI models and tools.

***

### W

Ahora vemos las respuestas que nos da el modelo utilizando las herramientas a las que le damos acceso.

In [ ]:
# Example 1: guest lookup
query = "Tell me about Lady Ada Lovelace. What's her background?"
response = await alfred.run(query)
print("🎩 Alfred's Response:")
print(response)

# Example 2: weather check
query = "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"
response = await alfred.run(query)
print("🎩 Alfred's Response:")
print(response)

# Example 3: Hub stats
query = "One of our guests is from Google. What can you tell me about their most popular model?"
response = await alfred.run(query)
print("🎩 Alfred's Response:")
print(response)

# Example 4: combining tools
query = "I need to speak with Dr. Nikola Tesla about recent advancements in wireless energy. Can you help me prepare for this conversation?"
response = await alfred.run(query)
print("🎩 Alfred's Response:")
print(response)

🎩 Alfred's Response:
Ada Lovelace (1815–1852) was an English mathematician and writer who is widely celebrated today as the **world's first computer programmer**. She is best known for her pioneering work on Charles Babbage’s proposed mechanical general-purpose computer, the **Analytical Engine**.

Here is an overview of her background and why she is such a significant figure in history.

### 1. Family Background
Ada’s life began with a dramatic separation of two very different worlds:
*   **The Poet Father:** She was the only legitimate child of the famous, volatile Romantic poet **Lord Byron** and his wife, **Annabella Milbanke**. Her parents separated just months after she was born, and Lord Byron left England shortly thereafter. Ada never knew her father personally.
*   **The Mother's Influence:** Fearing that Ada might inherit her father’s "dangerous" poetic temperament and mood swings, her mother, Lady Byron, insisted that Ada study mathematics and logic from a very young age. Th

Aquí evaluamos al agente para ver si utiliza las herramientas a las que tiene acceso correctamente junto al RAG.

In [ ]:
from llama_index.core.memory import Memory

memory = Memory.from_defaults(session_id="alfred_session", token_limit=30000)

# First interaction
response1 = await alfred.run("Tell me about Lady Ada Lovelace.", memory=memory)
print("🎩 Alfred's First Response:")
print(response1)

# Second interaction (referencing the first)
response2 = await alfred.run("What projects is she currently working on?", memory=memory)
print("🎩 Alfred's Second Response:")
print(response2)

🎩 Alfred's First Response:
Ada Lovelace (1815–1852) was a brilliant 19th-century English mathematician and writer, widely celebrated today as the **world's first computer programmer**.

She is best known for her visionary work on Charles Babbage’s early mechanical general-purpose computer, the **Analytical Engine**.

Here are the key aspects of her life and legacy:

### 1. Early Life and "Poetical Science"
Born Augusta Ada Byron on December 10, 1815, she was the only legitimate child of the famous Romantic poet Lord Byron and his wife, Anne Isabella Milbanke. Her parents separated shortly after her birth, and her mother, fearing that Ada might inherit her father’s "volatile" poetic temperament, insisted that she be tutored intensively in mathematics and logic.

Despite this emphasis on science, Ada did not lose her father's creative flair. She famously combined her analytical training with her imagination, calling her approach **"poetical science."** This unique perspective allowed her

Para finalizar, le damos al agente un componente de memoria para que recuerde las conversaciones anteriores y las incorpore en sus respuestas.

NOTA: Las diferencias de código entre este cuaderno y el original de Hugging Face se dieron a problemas de créditos apra utilizar los modelos propuestos. Se utilizó un modelo alterno de Google y el código fue cambiado con ayuda de la IA Claude sólo para este propósito. Adicionalmente el código también se cambió para utilizar Llama con su forma nativa de implementar RAGs en vez de LangChain.